
# COBYLA guiado por sensibilidade — preços → covariância → CVXPY → QUBO → Ising

Este notebook reconstrói integralmente o problema fixo de portfólio a partir de:

```python
pd.read_csv("../data/assets/stocks_prices.csv", index_col=0)
```

Configuração física/combinatória:

\[
n=10,\qquad k=4,\qquad q=0.5.
\]

A solução conhecida usada como auditoria é:

```text
1001011000
```

O pipeline executa:

1. leitura dos preços;
2. cálculo dos retornos por período;
3. construção do vetor de retornos médios;
4. construção da matriz de covariância;
5. solução clássica com CVXPY, com enumeração exata das 210 carteiras como verificação robusta;
6. transformação do problema em programa quadrático;
7. conversão para QUBO;
8. conversão do QUBO para Hamiltoniano de Ising;
9. construção do ansatz de Dicke para \(10\) qubits e peso de Hamming \(4\);
10. comparação do COBYLA completo com otimizações em subespaços ativos.

> O termo correto é **QUBO**, e não “cubo”: *Quadratic Unconstrained Binary Optimization*.



## Dependências

O ambiente precisa conter:

```python
# %pip install cvxpy qiskit-finance qiskit-optimization qiskit-algorithms
```

A classe `DickeStateAnsatz` continua sendo a implementação existente no seu projeto. O notebook tenta encontrá-la automaticamente dentro de `src/`.


In [ ]:

# ============================================================
# 1. IMPORTS E CONFIGURAÇÃO FIXA DO PROBLEMA
# ============================================================

from copy import deepcopy
from itertools import combinations
from pathlib import Path
from time import perf_counter
from typing import Any, Iterable
import importlib
import importlib.util
import json
import warnings

import cvxpy as cp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from qiskit.quantum_info import Statevector

try:
    from qiskit_finance.applications.optimization import PortfolioOptimization
except ImportError as exc:
    raise ImportError(
        "Instale qiskit-finance para construir o problema de portfólio."
    ) from exc

try:
    from qiskit_optimization.converters import QuadraticProgramToQubo
except ImportError as exc:
    raise ImportError(
        "Instale qiskit-optimization para converter o problema em QUBO."
    ) from exc

try:
    from qiskit_algorithms import NumPyMinimumEigensolver
    from qiskit_algorithms.optimizers import COBYLA
except ImportError:
    from qiskit.algorithms import NumPyMinimumEigensolver
    from qiskit.algorithms.optimizers import COBYLA


# Caminho solicitado, exatamente em minúsculo.
stock_path = Path("../data/assets/stocks_prices.csv")

# Problema fixo: 10 ativos, escolher 4.
n_assets = 10
budget = 4
risk_factor = 0.5

known_exact_bitstring = "1001011000"

# Configuração do experimento de subespaço ativo.
fast_mode = True

n_restarts = 3 if fast_mode else 20
maxiter_main = 250 if fast_mode else 1000
maxiter_refinement = 80 if fast_mode else 300

random_seed = 42
initial_perturbation_std = 0.15
near_optimal_threshold = 1e-3

output_dir = Path("results/cobyla_subespaco_ativo_v2")
output_dir.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(random_seed)

print("arquivo de preços:", stock_path)
print("número de ativos:", n_assets)
print("ativos escolhidos:", budget)
print("fator de risco q:", risk_factor)
print("bitstring conhecido:", known_exact_bitstring)
print("modo rápido:", fast_mode)


In [ ]:

# ============================================================
# 2. PREÇOS -> RETORNOS -> MATRIZ DE COVARIÂNCIA
# ============================================================

if not stock_path.exists():
    raise FileNotFoundError(
        "O arquivo não foi encontrado no caminho solicitado: "
        f"{stock_path.resolve()}"
    )

# Leitura solicitada: caminho em minúsculo e index_col=0.
stocks_prices = pd.read_csv(
    "../data/assets/stocks_prices.csv",
    index_col=0,
)

# Mantém somente colunas numéricas de preços.
stocks_prices = (
    stocks_prices
    .apply(pd.to_numeric, errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
    .ffill()
    .bfill()
    .dropna(axis=1, how="all")
    .dropna(axis=0, how="any")
)

if stocks_prices.shape[1] != n_assets:
    raise ValueError(
        f"Esperados exatamente {n_assets} ativos, mas o CSV possui "
        f"{stocks_prices.shape[1]} colunas numéricas."
    )

if len(stocks_prices) < 3:
    raise ValueError(
        "O CSV precisa possuir pelo menos três observações de preços."
    )

# Retornos simples entre períodos consecutivos.
period_returns = (
    stocks_prices
    .pct_change(fill_method=None)
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
)

# Mesma construção usada no problema de portfólio:
# vetor de retornos médios e matriz de covariância.
assets_return = (
    period_returns
    .mean(axis=0)
    .to_numpy(dtype=float)
)

covariance = (
    period_returns
    .cov()
    .to_numpy(dtype=float)
)

# Correção numérica mínima para garantir simetria.
covariance = 0.5 * (covariance + covariance.T)

# Pequenas autovalores negativas por arredondamento são removidas.
eigenvalues, eigenvectors = np.linalg.eigh(covariance)
eigenvalues_clipped = np.clip(eigenvalues, 0.0, None)

covariance_psd = (
    eigenvectors
    @ np.diag(eigenvalues_clipped)
    @ eigenvectors.T
)

covariance_psd = 0.5 * (
    covariance_psd + covariance_psd.T
)

tickers = list(stocks_prices.columns)

print("dimensão dos preços:", stocks_prices.shape)
print("dimensão dos retornos:", period_returns.shape)
print("dimensão do vetor de retornos:", assets_return.shape)
print("dimensão da covariância:", covariance_psd.shape)
print("menor autovalor da covariância:", np.linalg.eigvalsh(covariance_psd).min())

display(stocks_prices.head())
display(
    pd.DataFrame(
        covariance_psd,
        index=tickers,
        columns=tickers,
    )
)



## Função objetivo clássica

A função usada é a mesma estrutura do problema de portfólio do Qiskit:

\[
f(x)
=
q\,x^\top\Sigma x
-
\mu^\top x,
\]

sujeita a:

\[
\sum_{i=1}^{10}x_i=4,
\qquad
x_i\in\{0,1\}.
\]

Aqui:

- \(q=0.5\);
- \(\mu\) é o vetor de retornos médios;
- \(\Sigma\) é a matriz de covariância;
- \(x_i=1\) significa que o ativo foi selecionado.


In [ ]:

# ============================================================
# 3. SOLUÇÃO CLÁSSICA COM CVXPY
# ============================================================

x_cvx = cp.Variable(
    n_assets,
    boolean=True,
    name="portfolio_selection",
)

portfolio_objective_cvx = cp.Minimize(
    risk_factor * cp.quad_form(
        x_cvx,
        cp.psd_wrap(covariance_psd),
    )
    - assets_return @ x_cvx
)

portfolio_constraints_cvx = [
    cp.sum(x_cvx) == budget,
]

portfolio_problem_cvx = cp.Problem(
    portfolio_objective_cvx,
    portfolio_constraints_cvx,
)

# Solvers capazes de resolver MIQP binário.
miqp_priority = [
    "GUROBI",
    "CPLEX",
    "MOSEK",
    "SCIP",
    "XPRESS",
    "COPT",
    "ECOS_BB",
]

installed_solvers = set(cp.installed_solvers())
selected_miqp_solver = next(
    (
        solver
        for solver in miqp_priority
        if solver in installed_solvers
    ),
    None,
)

cvxpy_solution = None
cvxpy_objective = None
cvxpy_status = "solver_miqp_indisponivel"

if selected_miqp_solver is not None:
    try:
        portfolio_problem_cvx.solve(
            solver=selected_miqp_solver,
            verbose=False,
        )

        cvxpy_status = portfolio_problem_cvx.status

        if x_cvx.value is not None:
            cvxpy_solution = np.rint(
                np.asarray(x_cvx.value).reshape(-1)
            ).astype(int)

            cvxpy_objective = float(
                portfolio_problem_cvx.value
            )
    except Exception as exc:
        warnings.warn(
            "O solver MIQP foi localizado, mas não concluiu a solução: "
            f"{type(exc).__name__}: {exc}"
        )

print("solvers instalados:", sorted(installed_solvers))
print("solver MIQP selecionado:", selected_miqp_solver)
print("status CVXPY:", cvxpy_status)

if cvxpy_solution is not None:
    print("solução CVXPY em ordem dos ativos:", cvxpy_solution)
    print("objetivo CVXPY:", cvxpy_objective)


In [ ]:

# ============================================================
# 4. VERIFICAÇÃO EXATA DAS 210 CARTEIRAS
# ============================================================

def classical_portfolio_value(x_binary):
    """
    Avalia exatamente a mesma função clássica.
    """
    x_binary = np.asarray(
        x_binary,
        dtype=float,
    ).reshape(-1)

    return float(
        risk_factor
        * x_binary
        @ covariance_psd
        @ x_binary
        - assets_return
        @ x_binary
    )


enumeration_rows = []

for selected_indices in combinations(
    range(n_assets),
    budget,
):
    candidate = np.zeros(
        n_assets,
        dtype=int,
    )

    candidate[list(selected_indices)] = 1

    value = classical_portfolio_value(
        candidate
    )

    # A string de medição do Qiskit apresenta q_(n-1)...q_0.
    bitstring_asset_order = "".join(
        str(int(value_i))
        for value_i in candidate
    )

    bitstring_qiskit_order = (
        bitstring_asset_order[::-1]
    )

    enumeration_rows.append(
        {
            "selected_indices": selected_indices,
            "x_asset_order": candidate,
            "bitstring_asset_order": bitstring_asset_order,
            "bitstring_qiskit_order": bitstring_qiskit_order,
            "objective": value,
        }
    )

enumeration_df = (
    pd.DataFrame(enumeration_rows)
    .sort_values("objective")
    .reset_index(drop=True)
)

enumeration_best = enumeration_df.iloc[0]

x_enumeration = np.asarray(
    enumeration_best["x_asset_order"],
    dtype=int,
)

enumeration_objective = float(
    enumeration_best["objective"]
)

enumeration_asset_bitstring = str(
    enumeration_best["bitstring_asset_order"]
)

enumeration_qiskit_bitstring = str(
    enumeration_best["bitstring_qiskit_order"]
)

print("quantidade de carteiras avaliadas:", len(enumeration_df))
print("objetivo exato por enumeração:", enumeration_objective)
print("bitstring em ordem dos ativos:", enumeration_asset_bitstring)
print("bitstring em ordem do Qiskit:", enumeration_qiskit_bitstring)
print("bitstring conhecido:", known_exact_bitstring)

display(
    enumeration_df[
        [
            "selected_indices",
            "bitstring_asset_order",
            "bitstring_qiskit_order",
            "objective",
        ]
    ].head(10)
)

# Auditoria da solução CVXPY contra a enumeração.
if cvxpy_solution is not None:
    if not np.array_equal(
        cvxpy_solution,
        x_enumeration,
    ):
        warnings.warn(
            "A solução do solver CVXPY difere da enumeração exata. "
            "A enumeração será usada como referência final."
        )

# Detecta automaticamente qual convenção coincide com a string conhecida.
if enumeration_qiskit_bitstring == known_exact_bitstring:
    known_bitstring_convention = "qiskit"
elif enumeration_asset_bitstring == known_exact_bitstring:
    known_bitstring_convention = "asset"
else:
    known_bitstring_convention = "mismatch"

print("convenção do bitstring conhecido:", known_bitstring_convention)

if known_bitstring_convention == "mismatch":
    warnings.warn(
        "O problema reconstruído não retornou 1001011000. "
        "Verifique se o CSV, q=0.5 e a construção dos retornos "
        "são exatamente os mesmos usados no banco original."
    )


In [ ]:

# ============================================================
# 5. PROGRAMA QUADRÁTICO -> QUBO -> ISING
# ============================================================

portfolio_application = PortfolioOptimization(
    expected_returns=assets_return,
    covariances=covariance_psd,
    risk_factor=risk_factor,
    budget=budget,
)

quadratic_program = (
    portfolio_application
    .to_quadratic_program()
)

qubo_converter = QuadraticProgramToQubo()

qubo = qubo_converter.convert(
    quadratic_program
)

ising, offset = qubo.to_ising()

print("programa quadrático original:")
print(quadratic_program.prettyprint())

print()
print("QUBO:")
print(qubo.prettyprint())

print()
print("número de qubits do Ising:", ising.num_qubits)
print("offset do Ising:", offset)
print("número de termos de Pauli:", len(ising.paulis))


In [ ]:

# ============================================================
# 6. SOLUÇÃO EXATA DO HAMILTONIANO DE ISING
# ============================================================

exact_solver = NumPyMinimumEigensolver()

exact_result = exact_solver.compute_minimum_eigenvalue(
    operator=ising
)

exact_energy = float(
    np.real(
        exact_result.eigenvalue
        + offset
    )
)

exact_state_dict = (
    exact_result
    .eigenstate
    .to_dict()
)

exact_bitstring_ising = str(
    max(
        exact_state_dict,
        key=lambda key: abs(
            exact_state_dict[key]
        ) ** 2,
    )
).replace(" ", "")

print("energia exata do Ising + offset:", exact_energy)
print("bitstring exato do Ising:", exact_bitstring_ising)
print("bitstring conhecido:", known_exact_bitstring)
print(
    "coincide com o conhecido?",
    exact_bitstring_ising == known_exact_bitstring,
)

if exact_bitstring_ising != known_exact_bitstring:
    warnings.warn(
        "O Ising reconstruído não retornou o bitstring esperado. "
        "Não prossiga para conclusões de sensibilidade sem conferir "
        "a origem dos dados e a convenção de bits."
    )

# Referências usadas por todo o restante do notebook.
EXACT_ENERGY = exact_energy
EXACT_BITSTRING = known_exact_bitstring


In [ ]:

# ============================================================
# 7. LOCALIZAR A CLASSE DickeStateAnsatz DO PROJETO
# ============================================================

def resolve_dicke_state_ansatz():
    """
    Procura a classe já importada, módulos comuns e, por fim,
    arquivos Python dentro de src/.
    """
    if "DickeStateAnsatz" in globals():
        return globals()["DickeStateAnsatz"]

    module_candidates = [
        "src.ansatz.dicke_state_ansatz",
        "src.ansatzes.dicke_state_ansatz",
        "src.circuits.dicke_state_ansatz",
        "src.models.dicke_state_ansatz",
        "src.quantum.dicke_state_ansatz",
        "src.dicke_state_ansatz",
    ]

    for module_name in module_candidates:
        try:
            module = importlib.import_module(
                module_name
            )

            if hasattr(
                module,
                "DickeStateAnsatz",
            ):
                return module.DickeStateAnsatz
        except Exception:
            continue

    source_roots = [
        Path("../src"),
        Path("src"),
    ]

    for source_root in source_roots:
        if not source_root.exists():
            continue

        for python_file in source_root.rglob(
            "*.py"
        ):
            try:
                text = python_file.read_text(
                    encoding="utf-8",
                    errors="ignore",
                )
            except Exception:
                continue

            if "class DickeStateAnsatz" not in text:
                continue

            module_name = (
                "_dynamic_dicke_state_ansatz"
            )

            spec = importlib.util.spec_from_file_location(
                module_name,
                python_file,
            )

            if (
                spec is None
                or spec.loader is None
            ):
                continue

            module = importlib.util.module_from_spec(
                spec
            )

            try:
                spec.loader.exec_module(
                    module
                )
            except Exception:
                continue

            if hasattr(
                module,
                "DickeStateAnsatz",
            ):
                print(
                    "DickeStateAnsatz localizada em:",
                    python_file,
                )
                return module.DickeStateAnsatz

    raise ImportError(
        "A classe DickeStateAnsatz não foi localizada. "
        "Execute antes a célula de importação usada no notebook "
        "01_generate_dataset.ipynb ou ajuste module_candidates."
    )


DickeStateAnsatz = resolve_dicke_state_ansatz()

ansatz = DickeStateAnsatz(
    n=ising.num_qubits,
    k=budget,
).decompose()

ANSATZ = ansatz
N_PARAMS = int(ansatz.num_parameters)

if N_PARAMS != 30:
    raise ValueError(
        f"Esperados 30 parâmetros no ansatz, mas foram encontrados {N_PARAMS}."
    )

print("qubits do ansatz:", ansatz.num_qubits)
print("parâmetros do ansatz:", N_PARAMS)


In [ ]:

# ============================================================
# 8. FUNÇÃO DE ENERGIA E PONTO INICIAL
# ============================================================

parameter_order = list(
    ANSATZ.parameters
)

def bind_ansatz(theta):
    """
    Liga explicitamente cada valor à ordem dos parâmetros
    do circuito.
    """
    theta = np.asarray(
        theta,
        dtype=float,
    ).reshape(-1)

    if theta.size != N_PARAMS:
        raise ValueError(
            f"Esperados {N_PARAMS} parâmetros; recebidos {theta.size}."
        )

    parameter_map = {
        parameter: float(theta[index])
        for index, parameter in enumerate(
            parameter_order
        )
    }

    bound = ANSATZ.assign_parameters(
        parameter_map,
        inplace=False,
    )

    return bound.remove_final_measurements(
        inplace=False
    )


def exp_val(theta):
    """
    Valor esperado exato de H_Ising + offset.
    """
    state = Statevector.from_instruction(
        bind_ansatz(theta)
    )

    return float(
        np.real(
            state.expectation_value(
                ising
            )
        )
        + offset
    )


# Usa o ponto original, se ele já estiver definido no ambiente.
if (
    "initial_points_dict" in globals()
    and "Dicke_state" in initial_points_dict
):
    theta_initial = np.asarray(
        initial_points_dict[
            "Dicke_state"
        ][0],
        dtype=float,
    ).reshape(-1)
else:
    # Fallback reproduzível. Para o ansatz de Dicke, zero mantém
    # a preparação inicial e evita inserir informação externa.
    theta_initial = np.zeros(
        N_PARAMS,
        dtype=float,
    )

    initial_points_dict = {
        "Dicke_state": [
            theta_initial.copy()
        ]
    }

if theta_initial.size != N_PARAMS:
    raise ValueError(
        "O ponto inicial não possui 30 parâmetros."
    )

optimizers_dict = {
    "COBYLA": COBYLA(
        maxiter=maxiter_main,
        tol=1e-8,
    )
}

print("energia do ponto inicial:", exp_val(theta_initial))
print("forma do ponto inicial:", theta_initial.shape)


In [ ]:

# ============================================================
# 9. RANKING E CONJUNTOS ATIVOS
# ============================================================

def wrap_angles(theta):
    theta = np.asarray(
        theta,
        dtype=float,
    )

    return (
        theta + np.pi
    ) % (
        2 * np.pi
    ) - np.pi


# Ranking empírico, do maior para o menor contraste
# com o bitstring exato.
RANKED_INDICES = [
    17, 2, 14, 3, 24, 4, 18, 1, 16, 20,
    8, 6, 0, 19, 15, 28, 5, 29, 7, 9,
    23, 25, 10, 21, 13, 11, 12, 22, 26, 27,
]

if sorted(RANKED_INDICES) != list(
    range(N_PARAMS)
):
    raise ValueError(
        "RANKED_INDICES precisa conter 0...29 sem repetição."
    )

TOP1 = [17]
TOP2 = [17, 14]
TOP4 = [17, 2, 14, 3]
BOTTOM4 = [27, 26, 22, 12]

random_pool = [
    index
    for index in range(N_PARAMS)
    if index not in TOP4
]

RANDOM4 = sorted(
    rng.choice(
        random_pool,
        size=4,
        replace=False,
    ).tolist()
)

EXPERIMENTS = {
    "full_30": list(
        range(N_PARAMS)
    ),
    "top1_theta17": TOP1,
    "top2_theta17_theta14": TOP2,
    "top4_17_02_14_03": TOP4,
    "bottom4_control": BOTTOM4,
    "random4_control": RANDOM4,
}

theta_reference = wrap_angles(
    theta_initial.copy()
)

display(
    pd.DataFrame(
        [
            {
                "experimento": name,
                "n_ativos": len(indices),
                "indices": indices,
            }
            for name, indices in EXPERIMENTS.items()
        ]
    )
)


In [ ]:

# ============================================================
# 10. AVALIAÇÃO FÍSICA DE CADA SOLUÇÃO
# ============================================================

def scalar_energy(value: Any) -> float:
    return float(
        np.asarray(value).reshape(-1)[0]
    )


def evaluate_theta(theta):
    theta = wrap_angles(
        np.asarray(
            theta,
            dtype=float,
        ).reshape(-1)
    )

    energy = scalar_energy(
        exp_val(theta)
    )

    state = Statevector.from_instruction(
        bind_ansatz(theta)
    )

    probabilities = {
        str(bitstring).replace(
            " ",
            "",
        ): float(probability)
        for bitstring, probability
        in state.probabilities_dict().items()
    }

    dominant_bitstring = max(
        probabilities,
        key=probabilities.get,
    )

    return {
        "theta": theta,
        "energy": energy,
        "energy_gap": abs(
            energy - EXACT_ENERGY
        ),
        "exact_probability": probabilities.get(
            EXACT_BITSTRING,
            0.0,
        ),
        "dominant_bitstring": dominant_bitstring,
        "dominant_probability": probabilities[
            dominant_bitstring
        ],
        "is_exact_dominant": (
            dominant_bitstring
            == EXACT_BITSTRING
        ),
        "is_near_optimal": (
            abs(
                energy - EXACT_ENERGY
            )
            < near_optimal_threshold
        ),
    }


initial_check = evaluate_theta(
    theta_initial
)

print(
    "bitstring dominante no ponto inicial:",
    initial_check["dominant_bitstring"],
)

print(
    "P(bitstring exato) no ponto inicial:",
    initial_check["exact_probability"],
)


In [ ]:

# ============================================================
# 6. CONSTRUTOR DO COBYLA
# ============================================================

def make_cobyla(maxiter):
    """
    Preserva a configuração do COBYLA do projeto quando disponível.
    """
    if (
        "optimizers_dict" in globals()
        and "COBYLA" in optimizers_dict
    ):
        optimizer = deepcopy(optimizers_dict["COBYLA"])

        if hasattr(optimizer, "set_options"):
            try:
                optimizer.set_options(maxiter=int(maxiter))
            except Exception:
                pass

        return optimizer

    try:
        from qiskit_algorithms.optimizers import COBYLA
    except ImportError:
        from qiskit.algorithms.optimizers import COBYLA

    return COBYLA(maxiter=int(maxiter), tol=1e-8)


In [ ]:

# ============================================================
# 7. FUNÇÃO CENTRAL: COBYLA EM SUBESPAÇO ATIVO
# ============================================================

def reconstruct_full_theta(theta_active, active_indices, fixed_reference):
    """
    Reconstrói o vetor completo.
    Somente os índices ativos recebem os valores do COBYLA.
    """
    active_indices = np.asarray(list(active_indices), dtype=int)
    theta_active = np.asarray(theta_active, dtype=float).reshape(-1)

    if theta_active.size != active_indices.size:
        raise ValueError("Quantidade de valores e índices ativos não coincide.")

    theta_full = np.asarray(fixed_reference, dtype=float).copy()
    theta_full[active_indices] = theta_active

    return wrap_angles(theta_full)


def run_active_cobyla(
    active_indices: Iterable[int],
    fixed_reference,
    restart: int,
    method_name: str,
    maxiter: int,
    initial_full=None,
):
    active_indices = sorted(set(int(i) for i in active_indices))

    if initial_full is None:
        initial_full = np.asarray(fixed_reference, dtype=float).copy()

        initial_full[active_indices] = wrap_angles(
            initial_full[active_indices]
            + rng.normal(
                0.0,
                initial_perturbation_std,
                size=len(active_indices),
            )
        )
    else:
        initial_full = np.asarray(initial_full, dtype=float).copy()

    x0_active = initial_full[active_indices].copy()
    energy_history = []

    def restricted_objective(theta_active):
        theta_full = reconstruct_full_theta(
            theta_active,
            active_indices,
            fixed_reference,
        )

        energy = scalar_energy(exp_val(theta_full))
        energy_history.append(energy)

        return energy

    optimizer = make_cobyla(maxiter=maxiter)

    start = perf_counter()

    result = optimizer.minimize(
        fun=restricted_objective,
        x0=x0_active,
    )

    elapsed = perf_counter() - start

    theta_final = reconstruct_full_theta(
        result.x,
        active_indices,
        fixed_reference,
    )

    evaluation = evaluate_theta(theta_final)

    return {
        "method": method_name,
        "restart": restart,
        "active_indices": active_indices,
        "n_active": len(active_indices),
        "theta_initial": initial_full,
        "theta_final": theta_final,
        "energy_history": energy_history,
        "nfev": len(energy_history),
        "elapsed_s": elapsed,
        "evaluation": evaluation,
        "raw_result": result,
    }


In [ ]:

# ============================================================
# 8. SUÍTE PRINCIPAL
# ============================================================

all_runs = []

for method_name, active_indices in EXPERIMENTS.items():

    print()
    print("=" * 72)
    print(method_name, "| parâmetros ativos:", active_indices)
    print("=" * 72)

    for restart in range(n_restarts):

        run = run_active_cobyla(
            active_indices=active_indices,
            fixed_reference=theta_reference,
            restart=restart,
            method_name=method_name,
            maxiter=maxiter_main,
        )

        all_runs.append(run)

        ev = run["evaluation"]

        print(
            f"restart={restart:02d} | "
            f"nfev={run['nfev']:4d} | "
            f"gap={ev['energy_gap']:.6e} | "
            f"P*={ev['exact_probability']:.4f} | "
            f"exato={ev['is_exact_dominant']}"
        )


In [ ]:

# ============================================================
# 9. ESTRATÉGIA EM DOIS ESTÁGIOS
#    Top-4 -> refinamento completo curto
# ============================================================

for restart in range(n_restarts):

    stage1 = run_active_cobyla(
        active_indices=TOP4,
        fixed_reference=theta_reference,
        restart=restart,
        method_name="two_stage_top4",
        maxiter=maxiter_main,
    )

    stage2 = run_active_cobyla(
        active_indices=list(range(N_PARAMS)),
        fixed_reference=stage1["theta_final"],
        initial_full=stage1["theta_final"],
        restart=restart,
        method_name="top4_then_full_refinement",
        maxiter=maxiter_refinement,
    )

    stage2["energy_history"] = (
        stage1["energy_history"]
        + stage2["energy_history"]
    )

    stage2["nfev"] = stage1["nfev"] + stage2["nfev"]
    stage2["elapsed_s"] = stage1["elapsed_s"] + stage2["elapsed_s"]

    all_runs.append(stage2)

    ev = stage2["evaluation"]

    print(
        f"two-stage restart={restart:02d} | "
        f"nfev total={stage2['nfev']:4d} | "
        f"gap={ev['energy_gap']:.6e} | "
        f"P*={ev['exact_probability']:.4f}"
    )


In [ ]:

# ============================================================
# 10. DATAFRAME E RESUMO ESTATÍSTICO
# ============================================================

rows = []

for run in all_runs:
    ev = run["evaluation"]

    rows.append({
        "method": run["method"],
        "restart": run["restart"],
        "n_active": run["n_active"],
        "active_indices": json.dumps(run["active_indices"]),
        "nfev": run["nfev"],
        "elapsed_s": run["elapsed_s"],
        "energy": ev["energy"],
        "energy_gap": ev["energy_gap"],
        "exact_probability": ev["exact_probability"],
        "dominant_bitstring": ev["dominant_bitstring"],
        "dominant_probability": ev["dominant_probability"],
        "is_exact_dominant": ev["is_exact_dominant"],
        "is_near_optimal": ev["is_near_optimal"],
    })

results_df = pd.DataFrame(rows)

summary_df = (
    results_df
    .groupby(["method", "n_active"], as_index=False)
    .agg(
        runs=("restart", "count"),
        nfev_median=("nfev", "median"),
        nfev_mean=("nfev", "mean"),
        time_median_s=("elapsed_s", "median"),
        gap_median=("energy_gap", "median"),
        gap_mean=("energy_gap", "mean"),
        gap_std=("energy_gap", "std"),
        p_exact_median=("exact_probability", "median"),
        exact_dominant_rate=("is_exact_dominant", "mean"),
        near_optimal_rate=("is_near_optimal", "mean"),
    )
)

summary_df["exact_dominant_rate_pct"] = (
    100 * summary_df["exact_dominant_rate"]
)

summary_df["near_optimal_rate_pct"] = (
    100 * summary_df["near_optimal_rate"]
)

results_df.to_csv(output_dir / "runs.csv", index=False)
summary_df.to_csv(output_dir / "summary.csv", index=False)

display(summary_df.sort_values(["gap_median", "nfev_median"]))


In [ ]:

# ============================================================
# 11. GRÁFICOS PRINCIPAIS
# ============================================================

# Custo versus qualidade.
fig, ax = plt.subplots(figsize=(11, 6))

for method, group in results_df.groupby("method"):
    ax.scatter(
        group["nfev"],
        group["energy_gap"],
        s=70,
        alpha=0.75,
        label=method,
    )

ax.axhline(
    near_optimal_threshold,
    linestyle="--",
    linewidth=1.5,
    label="limite quase ótimo",
)

ax.set_yscale("log")
ax.set_xlabel("Avaliações da função objetivo")
ax.set_ylabel("|E final - E exata|")
ax.set_title("Custo de otimização versus qualidade energética")
ax.grid(alpha=0.25)
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
fig.savefig(
    output_dir / "01_custo_vs_gap.png",
    dpi=250,
    bbox_inches="tight",
)
plt.show()


# Número mediano de avaliações.
ordered = summary_df.sort_values("nfev_median")

fig, ax = plt.subplots(figsize=(12, 5.5))

bars = ax.barh(
    ordered["method"],
    ordered["nfev_median"],
)

for bar, value in zip(bars, ordered["nfev_median"]):
    ax.text(
        bar.get_width(),
        bar.get_y() + bar.get_height() / 2,
        f" {value:.0f}",
        va="center",
    )

ax.set_xlabel("Mediana de avaliações da função objetivo")
ax.set_title("Quantidade de interações do COBYLA")
ax.grid(axis="x", alpha=0.25)
fig.tight_layout()
fig.savefig(
    output_dir / "02_nfev_mediano.png",
    dpi=250,
    bbox_inches="tight",
)
plt.show()


# Taxa de bitstring exato.
rate_order = summary_df.sort_values("exact_dominant_rate_pct")

fig, ax = plt.subplots(figsize=(12, 5.5))

bars = ax.barh(
    rate_order["method"],
    rate_order["exact_dominant_rate_pct"],
)

for bar, value in zip(
    bars,
    rate_order["exact_dominant_rate_pct"],
):
    ax.text(
        bar.get_width(),
        bar.get_y() + bar.get_height() / 2,
        f" {value:.1f}%",
        va="center",
    )

ax.set_xlim(0, 105)
ax.set_xlabel("Execuções com bitstring exato dominante (%)")
ax.set_title("Taxa de acerto por estratégia")
ax.grid(axis="x", alpha=0.25)
fig.tight_layout()
fig.savefig(
    output_dir / "03_taxa_bitstring.png",
    dpi=250,
    bbox_inches="tight",
)
plt.show()


In [ ]:

# ============================================================
# 12. CURVAS DE CONVERGÊNCIA REPRESENTATIVAS
# ============================================================

representative = {}

for method in results_df["method"].unique():
    candidates = [run for run in all_runs if run["method"] == method]

    representative[method] = min(
        candidates,
        key=lambda run: run["evaluation"]["energy_gap"],
    )

fig, ax = plt.subplots(figsize=(13, 6))

for method, run in representative.items():

    history = np.asarray(run["energy_history"], dtype=float)
    best_so_far = np.minimum.accumulate(history)

    ax.plot(
        np.arange(1, len(best_so_far) + 1),
        best_so_far,
        linewidth=1.8,
        label=method,
    )

ax.axhline(
    EXACT_ENERGY,
    linestyle=":",
    linewidth=2,
    label="energia exata",
)

ax.set_xlabel("Avaliações acumuladas da função objetivo")
ax.set_ylabel("Melhor energia encontrada")
ax.set_title("Convergência representativa das estratégias")
ax.grid(alpha=0.25)
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
fig.savefig(
    output_dir / "04_convergencia.png",
    dpi=250,
    bbox_inches="tight",
)
plt.show()


In [ ]:

# ============================================================
# 13. CURVA DE DIMENSÃO ATIVA
# ============================================================

ACTIVE_DIMENSIONS = [1, 2, 4, 6, 8, 12, 16, N_PARAMS]

dimension_rows = []

for dimension in ACTIVE_DIMENSIONS:

    active_indices = RANKED_INDICES[:dimension]

    for restart in range(n_restarts):

        run = run_active_cobyla(
            active_indices=active_indices,
            fixed_reference=theta_reference,
            restart=restart,
            method_name=f"top_{dimension}",
            maxiter=maxiter_main,
        )

        ev = run["evaluation"]

        dimension_rows.append({
            "dimension": dimension,
            "restart": restart,
            "nfev": run["nfev"],
            "energy_gap": ev["energy_gap"],
            "exact_probability": ev["exact_probability"],
            "is_exact_dominant": ev["is_exact_dominant"],
        })

dimension_df = pd.DataFrame(dimension_rows)

dimension_summary = (
    dimension_df
    .groupby("dimension", as_index=False)
    .agg(
        gap_median=("energy_gap", "median"),
        nfev_median=("nfev", "median"),
        p_exact_median=("exact_probability", "median"),
        exact_rate=("is_exact_dominant", "mean"),
    )
)

dimension_df.to_csv(
    output_dir / "dimension_runs.csv",
    index=False,
)

dimension_summary.to_csv(
    output_dir / "dimension_summary.csv",
    index=False,
)

display(dimension_summary)

fig, ax = plt.subplots(figsize=(10, 5.5))

ax.plot(
    dimension_summary["dimension"],
    dimension_summary["gap_median"],
    marker="o",
    linewidth=2,
)

ax.axhline(
    near_optimal_threshold,
    linestyle="--",
    linewidth=1.5,
)

ax.set_yscale("log")
ax.set_xlabel("Quantidade de parâmetros ativos")
ax.set_ylabel("Gap energético mediano")
ax.set_title("Menor dimensão ativa necessária")
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(
    output_dir / "05_curva_dimensao.png",
    dpi=250,
    bbox_inches="tight",
)
plt.show()



## Como interpretar

Um resultado favorável ocorre quando o Top-4 apresenta:

\[
N_{\mathrm{fev}}^{\mathrm{Top4}}
\ll
N_{\mathrm{fev}}^{\mathrm{Completo}},
\]

mantendo gap energético, probabilidade do bitstring ótimo e taxa de acerto próximos ao COBYLA completo.

O controle `bottom4_control` e o `random4_control` precisam ser inferiores ao Top-4. Caso qualquer grupo de quatro parâmetros funcione igualmente bem, a vantagem pode vir apenas da redução dimensional.

A estratégia `top4_then_full_refinement` testa uma hipótese intermediária: os parâmetros mais sensíveis fazem a busca principal e os demais realizam apenas o refinamento final.

### Conclusões possíveis

- **Top-4 sozinho funciona:** há evidência de um subespaço ativo útil.
- **Top-4 falha, mas o refinamento funciona:** os parâmetros sensíveis orientam a busca, mas os demais são necessários para ajuste fino.
- **Top-4 e controles são semelhantes:** a seleção empírica não demonstrou vantagem.
- **Somente o completo funciona:** os parâmetros são sensíveis, mas não suficientes devido a interações e compensações.

A primeira conclusão é válida apenas para este Hamiltoniano, este ansatz e este problema fixo. A generalização exige repetir o teste com outras matrizes de retorno e covariância, outros ativos, valores de \(k\), sementes e ansätze.
